# Import libraries

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

load_dotenv()

True

# Get existing vectorstore

In [2]:
api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

In [3]:
# Pull all documents from the vectorstore
all_docs = vectorstore.get(include=['documents', 'metadatas'])

# Reconstruct Document objects for BM25
docs_from_chroma = [
    Document(page_content=text, metadata=meta) 
    for text, meta in zip(all_docs['documents'], all_docs['metadatas'])
]

In [19]:
print(f'Length: {len(docs_from_chroma)} \n')
print(f'Keys: {vars(docs_from_chroma[50]).keys()}')
print(f'Metadata keys: {vars(docs_from_chroma[50])["metadata"].keys()}')
print(f'Metadata content: {vars(docs_from_chroma[50])["metadata"]}')
print(f'Page content: {vars(docs_from_chroma[50])["page_content"][:20]}')

Length: 186 

Keys: dict_keys(['id', 'metadata', 'page_content', 'type'])
Metadata keys: dict_keys(['talks_about_tokenization', 'talks_about_language_modeling', 'content_type', 'linguistic_focus', 'chapter', 'creationdate', 'contains_regex', 'page', 'contains_code_or_cli', 'total_pages', 'talks_about_morphology', 'contains_table', 'talks_about_ngrams', 'importance_score', 'source', 'contains_math_latex', 'volume'])
Metadata content: {'talks_about_tokenization': True, 'talks_about_language_modeling': False, 'content_type': 'Narrative', 'linguistic_focus': 'English, Chinese, Japanese, Thai', 'chapter': 2, 'creationdate': 'D:20260329110007', 'contains_regex': False, 'page': 2, 'contains_code_or_cli': False, 'total_pages': 34, 'talks_about_morphology': False, 'contains_table': True, 'talks_about_ngrams': False, 'importance_score': 'High', 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_math_latex': False, 'volume': 1}
Page content: 6
CHAPTER 2
•
WORDS 


# Initialize and use single retrievers

### BM25

In [7]:
bm25_retriever = BM25Retriever.from_documents(docs_from_chroma)
bm25_retriever.k = 2

In [17]:
bm25_retriever.invoke("What is BPE?")

[Document(metadata={'contains_table': False, 'linguistic_focus': '', 'total_pages': 34, 'talks_about_morphology': False, 'importance_score': 'High', 'contains_code_or_cli': False, 'content_type': 'Narrative', 'contains_math_latex': False, 'talks_about_tokenization': False, 'creationdate': 'D:20260329110007', 'chapter': 2, 'talks_about_ngrams': False, 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_regex': True, 'talks_about_language_modeling': False, 'page': 15, 'volume': 1}, page_content='2.6\n•\nREGULAR EXPRESSIONS\n19\nLanguage variety: What language (including dialect/region) was the corpus in?\nSpeaker demographics: What was, e.g., the age or gender of the text’s authors?\nCollection process: How big is the data? If it is a subsample how was it sampled?\nWas the data collected with consent? How was the data pre-processed, and\nwhat metadata is available?\nAnnotation process: What are the annotations, what are the demographics of the\nannotators, how were they trained, how wa

### Vector retriever

In [8]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [18]:
vector_retriever.invoke("What is BPE?")

[Document(id='55a6a0a4-4c49-5d5e-bd2b-e6dd375a906c', metadata={'creationdate': 'D:20260329110007', 'contains_regex': False, 'talks_about_tokenization': True, 'talks_about_language_modeling': False, 'importance_score': 'High', 'linguistic_focus': '', 'talks_about_morphology': False, 'contains_math_latex': False, 'source': 'data\\raw\\book_chapter_02.pdf', 'total_pages': 34, 'page': 12, 'contains_table': False, 'content_type': 'Narrative', 'contains_code_or_cli': False, 'talks_about_ngrams': False, 'chapter': 2, 'volume': 1}, page_content='2.4.3\nBPE in practice\nThe example above just showed simple BPE learning from sequences of ASCII\nbytes. How does BPE work with Unicode input? We normally run BPE on the\nindividual bytes of UTF-8-encoded text. That is, we take a Unicode representations\nof text as a series of code points, encode it in bytes using UTF-8, and we treat each of\nthese individual bytes as the input to BPE. Thus BPE likely begins by rediscovering\nthe 2-byte and common 3-b

# Ensemble retriever

In [9]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5]
)

## Use ensemble retriever

In [10]:
ensemble_retriever.invoke("What is BPE?")

[Document(metadata={'contains_table': False, 'linguistic_focus': '', 'total_pages': 34, 'talks_about_morphology': False, 'importance_score': 'High', 'contains_code_or_cli': False, 'content_type': 'Narrative', 'contains_math_latex': False, 'talks_about_tokenization': False, 'creationdate': 'D:20260329110007', 'chapter': 2, 'talks_about_ngrams': False, 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_regex': True, 'talks_about_language_modeling': False, 'page': 15, 'volume': 1}, page_content='2.6\n•\nREGULAR EXPRESSIONS\n19\nLanguage variety: What language (including dialect/region) was the corpus in?\nSpeaker demographics: What was, e.g., the age or gender of the text’s authors?\nCollection process: How big is the data? If it is a subsample how was it sampled?\nWas the data collected with consent? How was the data pre-processed, and\nwhat metadata is available?\nAnnotation process: What are the annotations, what are the demographics of the\nannotators, how were they trained, how wa